In [1]:
# Import packages and GDrive Driver for Colab to access file
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [4]:
# Import data

PATH = "/content/drive/MyDrive/healthcare_access_final.csv"
df = pd.read_csv(PATH)
df.head()

,fips,state,county,population,pop_density,poverty_rate,median_income,unemployment_rate,uninsured_rate,medicaid_rate,...,dist_clinic_clean,dist_ed_clean,is_hpsa,official_shortage_designation,expected_shortage_risk,risk_tier,access_gap,access_gap_index,access_alignment,priority_review_flag
0,1001,Alabama,Autauga County,59285,99.73,11.7,69841.0,2.2,7.36,12.04,...,8.96,5.29,1,Yes,0.264986,Low,0.735014,86.6,Officially Designated,0
1,1007,Alabama,Bibb County,22152,35.59,19.4,51215.0,2.5,8.32,20.28,...,3.88,8.34,1,Yes,0.423688,Moderate,0.576312,77.9,Officially Designated,0
2,1009,Alabama,Blount County,59292,91.94,12.8,61096.0,2.1,10.19,17.00,...,4.04,9.70,1,Yes,0.416113,Moderate,0.583887,78.3,Officially Designated,0
3,1023,Alabama,Choctaw County,12525,13.71,24.8,44483.0,4.0,8.17,20.16,...,4.80,10.84,1,Yes,0.302058,Moderate,0.697942,84.6,Officially Designated,0
4,1027,Alabama,Clay County,14188,23.49,16.9,51852.0,2.4,8.42,18.55,...,5.32,6.78,1,Yes,0.565275,Elevated,0.434725,70.0,Officially Designated,0


In [8]:
# Keep only priority review counties
priority = df[df["access_alignment"] == "High-Risk Not Designated"].copy()

# Create interpretable inputs
priority["provider_scarcity"] = 1 / (priority["physician_office_rate_clean"] + 0.001)
priority["distance_to_care"] = priority["dist_clinic_clean"]
priority["population_scale"] = np.log1p(priority["population"])

# Normalize components
scaler = MinMaxScaler()

priority[[
    "risk_component",
    "provider_scarcity_component",
    "distance_component",
    "population_component"
]] = scaler.fit_transform(priority[[
    "expected_shortage_risk",
    "provider_scarcity",
    "distance_to_care",
    "population_scale"
]])

# Weighted priority score
priority["priority_score"] = (
    priority["risk_component"] * 0.40 +
    priority["provider_scarcity_component"] * 0.30 +
    priority["distance_component"] * 0.20 +
    priority["population_component"] * 0.10
)

priority["priority_score"] = (priority["priority_score"] * 100).round(1)

# Rank counties
priority["priority_rank"] = priority["priority_score"].rank(
    ascending=False,
    method="dense"
).astype(int)

priority = priority.sort_values("priority_rank")

# Priority tier
priority["priority_tier"] = pd.cut(
    priority["priority_rank"],
    bins=[0, 10, 20, len(priority)],
    labels=["Top 10 Priority", "Top 20 Priority", "Remaining Priority Counties"]
)

#Recommendation Layer

def generate_recommendation(row):
    reasons = []

    if row["expected_shortage_risk"] >= 0.75:
        reasons.append(f"very high expected access risk ({row['expected_shortage_risk']:.1%})")
    else:
        reasons.append(f"elevated expected access risk ({row['expected_shortage_risk']:.1%})")

    if row["physician_office_rate_clean"] == 0:
        reasons.append("no recorded physician offices per capita")
    elif row["physician_office_rate_clean"] * 10 < 1:
        reasons.append(f"low physician office availability ({row['physician_office_rate_clean'] * 10:.1f} per 10,000 residents)")

    if row["dist_clinic_clean"] >= 10:
        reasons.append(f"long distance to clinic ({row['dist_clinic_clean']:.1f} miles)")
    elif row["dist_clinic_clean"] >= 5:
        reasons.append(f"above-average distance to clinic ({row['dist_clinic_clean']:.1f} miles)")

    if row["uninsured_rate"] >= 10:
        reasons.append(f"higher uninsured rate ({row['uninsured_rate']:.1f}%)")

    reason_text = ", ".join(reasons)

    return (
        f"{reason_text}. Recommend priority review of local provider capacity, "
        f"clinic access, and shortage designation criteria."
    )

priority["intervention_recommendation"] = priority.apply(generate_recommendation, axis=1)

# Export
OUT_DIR = "/content/drive/MyDrive"
tableau_path = f"{OUT_DIR}/priority_counties_for_intervention.csv"
priority.to_csv(tableau_path, index=False)


priority[[
    "priority_rank",
    "county",
    "state",
    "priority_score",
    "expected_shortage_risk",
    "physician_office_rate_clean",
    "dist_clinic_clean",
    "population",
    "intervention_recommendation"
]].head(20)

,priority_rank,county,state,priority_score,expected_shortage_risk,physician_office_rate_clean,dist_clinic_clean,population,intervention_recommendation
1728,1,McPherson County,Nebraska,80.0,0.750529,0.0,42.92,463,"very high expected access risk (75.1%), no rec..."
887,2,Menard County,Illinois,70.6,0.789246,0.0,2.53,12169,"very high expected access risk (78.9%), no rec..."
1024,3,Monroe County,Iowa,65.3,0.748645,0.0,3.36,7546,"elevated expected access risk (74.9%), no reco..."
1689,4,Arthur County,Nebraska,64.6,0.676375,0.0,28.96,540,"elevated expected access risk (67.6%), no reco..."
1697,4,Cherry County,Nebraska,64.6,0.679981,0.0,20.45,5468,"elevated expected access risk (68.0%), no reco..."
2858,5,Kane County,Utah,61.9,0.698951,0.0,8.72,7996,"elevated expected access risk (69.9%), no reco..."
1758,6,Humboldt County,Nevada,57.3,0.651687,0.0,8.62,17299,"elevated expected access risk (65.2%), no reco..."
1112,7,Osage County,Kansas,57.0,0.665932,0.0,4.59,15780,"elevated expected access risk (66.6%), no reco..."
2895,8,Dinwiddie County,Virginia,56.5,0.649208,0.0,5.87,28083,"elevated expected access risk (64.9%), no reco..."
3032,8,Bayfield County,Wisconsin,56.5,0.637361,0.0,10.72,16410,"elevated expected access risk (63.7%), no reco..."
